# Improving performance

Baseline used simple CNN model. 
We will make following changes:
1. Apply various image transformations (data augmentation)
2. Deeper CNN
3. Better optimizer
4. Increase training epoch #

Flow:
1. Seed fix, GPU setup
2. data prep
3. model creation (deep CNN)
4. model training
5. eval
6. prediction/sub



## 1. Fix Seed, GPU setup

In [2]:
import torch, random, os
import numpy as np

# Seed fixing
seed = 50
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
torch.manual_seed(seed)         # using CPU
torch.cuda.manual_seed(seed)    # using GPU
torch.cuda.manual_seed_all(seed) # multi GPU

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False

In [3]:
# Setting up GPU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## 2. Data prep

In [4]:
import pandas as pd

data_path = '/kaggle/input/aerial-cactus-identification/'
labels = pd.read_csv(data_path + 'train.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/aerial-cactus-identification/train.csv'

In [ ]:
# extract images
from zipfile import ZipFile

with ZipFile(data_path + 'train.zip') as zipper:
    zipper.extractall()

with ZipFile(data_path + 'test.zip') as zipper:
    zipper.extractall()

In [ ]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(labels,
                                test_size=0.1,
                                stratify = labels['has_cactus'],
                                random_state = 50)

print("num train data: ", len(train))
print("num valid data: ", len(valid))

In [ ]:
import cv2
from torch.utils.data import Dataset

class ImageDataset(Dataset):

    def __init__(self, df, img_dir = './', transform=None):
        super().__init__() # Call inhereted dataset generator
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self): # redefine dataset size
        return len(self.df)

    def __getitem__(self,idx):  # return data on designated index
        img_id = self.df.iloc[idx, 0]    # image ID
        img_path = self.img_dir + img_id # image path
        image = cv2.imread(img_path)     
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # fix color
        label = self.df.iloc[idx, 1]    # get label

        if self.transform is not None:
            image = self.transform(image)
        
        return image, label



### generating dataset / Image transfomation

In [ ]:
# Image transformaton
from torchvision import transforms 



transform_train = transforms.Compose([transforms.ToTensor(),    # ndarray -> tensor
                                      transforms.Pad(32, padding_mode='symmetric'), # add padding around image
                                      transforms.RandomHorizontalFlip(),    # defaults 50% random pick
                                      transforms.RandomVerticalFlip(),  # defaults 50% random pick
                                      transforms.RandomRotation(10),    # -10 ~ 10deg random rot
                                      transforms.Normalize((0.485, 0.456, 0.406),
                                                           (0.229, 0.224, 0.225))]) # normalize tensor form of images

transform_test= transforms.Compose([transforms.ToTensor(),
                                    transforms.Pad(32, padding_mode='symmetric'),   
                                    transforms.Normalize((0.485, 0.456, 0.406), # mean (RGB)
                                                         (0.229, 0.224, 0.225))])   # stdev (RGB)

In [ ]:
dataset_train = ImageDataset(df=labels, img_dir='train/', transform=transform_train)
dataset_valid = ImageDataset(df=valid, img_dir='train/', transform=transform_test)

### Data loader

In [ ]:
# For data loader multi processing
def seed_worker(worker_id):
    # def seed fixing func, fix generator seed value
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(0)

    

In [ ]:
# Data loader --> load data per defined batch size
from torch.utils.data import DataLoader
loader_train = DataLoader(dataset = dataset_train, batch_size = 32, shuffle=True,    # shuffle for training
                        #   worker_init_fn = seed_worker,
                        #   generator = g,
                        #   num_workers = 2
                          ) 
loader_valid = DataLoader(dataset = dataset_valid, batch_size = 32, shuffle=False,
                        #   worker_init_fn = seed_worker, 
                        #   generator = g,
                        #   num_workers = 2
                          )

# set batch size as power of 2.. per Ibrahem Kandel paper (2020)

## 3. Model generation
1. Input 3x32x32
2. CNN / batch norm / max pooling
3. CNN / batch norm / max pooling
4. CNN / batch norm / max pooling
5. CNN / batch norm / max pooling
6. CNN / batch norm / max pooling
7. Avg pooling
8. Flatten
9. linear layer
10. linear layer
11. output

Activation function = leaky relu





In [ ]:
import torch.nn as nn 
import torch.nn.functional as F 

class Model(nn.Module):

    def __init__(self):
        super().__init__() 
        

        # Layer 1-5: {Convolution, Batch Normalization, Max Pooling}
        self.layer1 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=32,
                                              kernel_size=3, padding=2),
                                    nn.BatchNorm2d(32), # batch normalization
                                    nn.LeakyReLU(), # LeakyReLU Activation func
                                    nn.MaxPool2d(kernel_size=2))

        

        self.layer2 = nn.Sequential(nn.Conv2d(in_channels=32, out_channels=64,
                                              kernel_size=3, padding=2),
                                    nn.BatchNorm2d(64),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))
        
        

        self.layer3 = nn.Sequential(nn.Conv2d(in_channels=64, out_channels=128,
                                              kernel_size=3, padding=2),
                                    nn.BatchNorm2d(128),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))
        
        

        self.layer4 = nn.Sequential(nn.Conv2d(in_channels=128, out_channels=256,
                                              kernel_size=3, padding=2),
                                    nn.BatchNorm2d(256),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))
        
        

        self.layer5 = nn.Sequential(nn.Conv2d(in_channels=256, out_channels=512,
                                              kernel_size=3, padding=2),
                                    nn.BatchNorm2d(512),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))
        
        


        # Avg pooling layer 
        self.avg_pool = nn.AvgPool2d(kernel_size=4) 

        # Fully connected layer 
        self.fc1 = nn.Linear(in_features=512 * 1 * 1, out_features=64)
        self.fc2 = nn.Linear(in_features=64, out_features=2)

    # Forward pass output
    def forward(self, x):   # (32,3,96,96)
        x = self.layer1(x)  # (32,32,49,49)
        x = self.layer2(x)  # (32,64,25,25)
        x = self.layer3(x)  # (32,128,13,13)
        x = self.layer4(x)  # (32,256,7,7)
        x = self.layer5(x)  # (32,512,4,4)
        x = self.avg_pool(x)    # (32,512,1,1)
        x = x.view(-1, 512 * 1 * 1) # Flattening -> (32,512)
        x = self.fc1(x) # (32,64)
        x = self.fc2(x) # (32,2)
        return x

In [ ]:
model = Model().to(device)

## 4. Model Training

In [5]:
# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adamax(model.parameters(), lr=0.00006)

# chagnes from baseline model: adamax instead of SGD, much less lr



NameError: name 'nn' is not defined

In [ ]:
# Increased epoch form 10 to 70

epochs = 70

# Repeat for the total number of epochs
for epoch in range(epochs):
    epoch_loss = 0 # initialize loss value for each epoch
    
    # Repeat for the number of iterations (batches)
    for images, labels in loader_train:
        # Move image and label mini-batch data to device
        images = images.to(device)
        labels = labels.to(device)
        
        # Reset gradient in optimizer
        optimizer.zero_grad()
        # Forward pass: use image as input
        outputs = model(images)
        # Compute loss between outputs and labels using loss func
        loss = criterion(outputs, labels)
        # Accumulate loss for curr batch
        epoch_loss += loss.item() 

        # Backprop:
        # Gradients are assigned to NN's weight (params) based on loss value
        loss.backward()

        # Update weights
        # New weight = oldweight - (lr * gradient)
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss/len(loader_train):.4f}')    
